# RSNA Pneumonia Detection: Step 1 - Patient-Level Preprocessing & Caching
**Project:** Foundation-Model-Based ROI Refinement of RT-DETR Predictions for False-Positive Reduction  
**Primary Detector:** RT-DETR-L (Stage 1 Proposal Generator)  
**Deliverables:**
1. Zero Patient Leakage Splitting (70% Train / 15% Val / 15% Test grouped strictly by `patientId`).
2. Medical DICOM standard normalization (`PhotometricInterpretation`, 8-bit scaling, 640x640 or 1024x1024 resizing).
3. YOLO Bounding Box annotation files (`0 xc yc nw nh`).
4. Multi-Level Caching (`rsna_pneumonia.yaml` and `rsna_rtdetr_dataset.zip` for instant reuse across sessions).


In [1]:
# Cell 1: Install required dependencies
!pip install -q pydicom opencv-python scikit-learn pandas numpy


In [2]:
# Cell 2: Imports & Dynamic Kaggle Path Resolution
import os
import sys
import json
import shutil
import cv2
import numpy as np
import pandas as pd
import pydicom
from concurrent.futures import ProcessPoolExecutor
from sklearn.model_selection import train_test_split

def resolve_dataset_paths():
    """Automatically locates the RSNA competition dataset on Kaggle."""
    candidate_inputs = [
        "/kaggle/input/competitions/rsna-pneumonia-detection-challenge",
        "/kaggle/input/rsna-pneumonia-detection-challenge",
        os.path.join(os.getcwd(), "rsna-pneumonia-detection-challenge"),
        os.getcwd()
    ]
    for cand in candidate_inputs:
        labels_check = os.path.join(cand, "stage_2_train_labels.csv")
        if os.path.exists(labels_check):
            return cand
    return "/kaggle/input/competitions/rsna-pneumonia-detection-challenge"

DATASET_DIR = resolve_dataset_paths()
TRAIN_IMAGE_DIR = os.path.join(DATASET_DIR, "stage_2_train_images")
LABEL_PATH = os.path.join(DATASET_DIR, "stage_2_train_labels.csv")
CLASS_INFO_PATH = os.path.join(DATASET_DIR, "stage_2_detailed_class_info.csv")
OUTPUT_DIR = "/kaggle/working/rsna_rtdetr_dataset"

print(f"[CONFIG] Base Dataset Path : {DATASET_DIR}")
print(f"[CONFIG] Raw DICOM Images  : {TRAIN_IMAGE_DIR}")
print(f"[CONFIG] Labels CSV Path   : {LABEL_PATH}")
print(f"[CONFIG] Detailed Class CSV: {CLASS_INFO_PATH}")
print(f"[CONFIG] Output Directory  : {OUTPUT_DIR}")


[CONFIG] Base Dataset Path : /kaggle/input/competitions/rsna-pneumonia-detection-challenge
[CONFIG] Raw DICOM Images  : /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_images
[CONFIG] Labels CSV Path   : /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_train_labels.csv
[CONFIG] Detailed Class CSV: /kaggle/input/competitions/rsna-pneumonia-detection-challenge/stage_2_detailed_class_info.csv
[CONFIG] Output Directory  : /kaggle/working/rsna_rtdetr_dataset


## 2. Zero-Leakage Patient-Level Splitting

In the RSNA dataset, many patients received multiple radiographs over time. Random image splitting causes radiographs from the same patient to appear in both training and testing partitions, leading to **severe patient identity leakage**.

We group annotations strictly by `patientId`, stratify on max `Target`, and assert mathematically:
$$\text{Train} \cap \text{Val} = \emptyset, \quad \text{Train} \cap \text{Test} = \emptyset, \quad \text{Val} \cap \text{Test} = \emptyset$$


In [3]:
# Cell 3: Zero-Leakage Grouped Splitting Logic
def load_and_merge_metadata(label_path, class_info_path):
    """Loads bounding box labels and detailed radiological classes, merging on patientId."""
    df_labels = pd.read_csv(label_path)
    df_class = pd.read_csv(class_info_path)
    df_class_unique = df_class.drop_duplicates(subset=["patientId"])
    df_merged = pd.merge(df_labels, df_class_unique, on="patientId", how="left")
    
    print(f"Total annotation rows: {len(df_merged)}")
    print(f"Total unique patients: {df_merged['patientId'].nunique()}")
    print(f"Pneumonia positive   : {(df_merged['Target'] == 1).sum()} rows")
    print(f"Pneumonia negative   : {(df_merged['Target'] == 0).sum()} rows")
    return df_merged


def create_patient_level_splits(df_merged, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, random_seed=42):
    """Partitions unique patients into Train, Val, and Test with zero patient leakage."""
    assert np.isclose(train_ratio + val_ratio + test_ratio, 1.0), "Split ratios must sum to 1.0"

    # Aggregate target per patient (1 if patient has pneumonia, else 0)
    patient_summary = df_merged.groupby("patientId").agg({
        "Target": "max",
        "class": "first"
    }).reset_index()

    patient_ids = np.array(patient_summary["patientId"].tolist())
    patient_targets = np.array(patient_summary["Target"].tolist(), dtype=np.int64)

    # First split: Separate Train from (Val + Test)
    temp_ratio = val_ratio + test_ratio
    train_patients, temp_patients, train_targets, temp_targets = train_test_split(
        patient_ids,
        patient_targets,
        test_size=temp_ratio,
        stratify=patient_targets,
        random_state=random_seed
    )

    # Second split: Separate Val from Test evenly
    relative_test_ratio = test_ratio / temp_ratio
    val_patients, test_patients, _, _ = train_test_split(
        temp_patients,
        temp_targets,
        test_size=relative_test_ratio,
        stratify=temp_targets,
        random_state=random_seed
    )

    # Mathematical Zero Leakage Assertions
    train_set = set(train_patients)
    val_set = set(val_patients)
    test_set = set(test_patients)
    assert len(train_set.intersection(val_set)) == 0, "CRITICAL: Patient leakage between Train and Val!"
    assert len(train_set.intersection(test_set)) == 0, "CRITICAL: Patient leakage between Train and Test!"
    assert len(val_set.intersection(test_set)) == 0, "CRITICAL: Patient leakage between Val and Test!"

    print("\n--- Patient-Level Split Summary (Zero Leakage Verified) ---")
    print(f"Train patients      : {len(train_patients):>5} ({len(train_patients)/len(patient_ids)*100:.1f}%)")
    print(f"Validation patients : {len(val_patients):>5} ({len(val_patients)/len(patient_ids)*100:.1f}%)")
    print(f"Test patients       : {len(test_patients):>5} ({len(test_patients)/len(patient_ids)*100:.1f}%)")
    print("Zero Leakage Check  : Train ∩ Val = 0, Train ∩ Test = 0, Val ∩ Test = 0 [PASSED]")

    split_map = {pid: "train" for pid in train_patients}
    split_map.update({pid: "val" for pid in val_patients})
    split_map.update({pid: "test" for pid in test_patients})

    df_merged["split"] = df_merged["patientId"].map(split_map)
    return df_merged


## 3. DICOM Preprocessing & YOLO Label Formatting

- **Photometric Interpretation:** Inverts `MONOCHROME1` to `MONOCHROME2` (so bones and fluid consolidations are white).
- **8-bit Normalization:** Standardizes floating-point pixel arrays to uint8 [0, 255].
- **YOLO Label Format:** Normalized coordinates:
  $$x_{\text{center}} = \frac{x + w/2}{\text{orig\_w}}, \quad y_{\text{center}} = \frac{y + h/2}{\text{orig\_h}}, \quad w_{\text{norm}} = \frac{w}{\text{orig\_w}}, \quad h_{\text{norm}} = \frac{h}{\text{orig\_h}}$$


In [4]:
# Cell 4: DICOM Reading & YOLO Box Conversion
def read_and_process_dicom(dcm_path, target_size=640):
    """Reads DICOM, handles MONOCHROME1 inversion, min-max normalizes, and resizes."""
    try:
        dcm = pydicom.dcmread(dcm_path)
        pixel_array = dcm.pixel_array.astype(np.float32)

        # Invert MONOCHROME1
        if getattr(dcm, "PhotometricInterpretation", "") == "MONOCHROME1":
            pixel_array = np.amax(pixel_array) - pixel_array

        # Min-max normalization
        p_min, p_max = pixel_array.min(), pixel_array.max()
        if p_max > p_min:
            pixel_array = (pixel_array - p_min) / (p_max - p_min) * 255.0
        else:
            pixel_array = np.zeros_like(pixel_array)

        image_uint8 = pixel_array.astype(np.uint8)
        orig_h, orig_w = image_uint8.shape

        if (orig_w, orig_h) != (target_size, target_size):
            image_resized = cv2.resize(image_uint8, (target_size, target_size), interpolation=cv2.INTER_LINEAR)
        else:
            image_resized = image_uint8

        image_rgb = cv2.cvtColor(image_resized, cv2.COLOR_GRAY2RGB)
        return image_rgb, orig_w, orig_h

    except Exception as e:
        print(f"Error reading DICOM {dcm_path}: {e}")
        return None, 1024, 1024


def convert_box_to_yolo_format(x, y, w, h, orig_w=1024, orig_h=1024):
    """Converts pixel bounding boxes to normalized [0, 1] YOLO coordinates."""
    xc = np.clip((x + w / 2.0) / orig_w, 0.0, 1.0)
    yc = np.clip((y + h / 2.0) / orig_h, 0.0, 1.0)
    nw = np.clip(w / orig_w, 0.0, 1.0)
    nh = np.clip(h / orig_h, 0.0, 1.0)
    return xc, yc, nw, nh


def process_single_patient(args):
    """Parallel worker for extracting a single patient's radiograph and labels."""
    patient_id, patient_rows, input_dir, output_dir, target_size = args
    split = patient_rows.iloc[0]["split"]
    dcm_path = os.path.join(input_dir, f"{patient_id}.dcm")

    img_save_path = os.path.join(output_dir, "images", split, f"{patient_id}.jpg")
    label_save_path = os.path.join(output_dir, "labels", split, f"{patient_id}.txt")

    if os.path.exists(img_save_path) and os.path.exists(label_save_path):
        return True

    image_rgb, orig_w, orig_h = read_and_process_dicom(dcm_path, target_size=target_size)
    if image_rgb is None:
        return False

    cv2.imwrite(img_save_path, image_rgb, [int(cv2.IMWRITE_JPEG_QUALITY), 95])

    label_lines = []
    for _, row in patient_rows.iterrows():
        if row["Target"] == 1:
            xc, yc, nw, nh = convert_box_to_yolo_format(row["x"], row["y"], row["width"], row["height"], orig_w, orig_h)
            label_lines.append(f"0 {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}\n")

    with open(label_save_path, "w") as f:
        f.writelines(label_lines)

    return True


## 4. Multi-Level Caching & Execution Engine

This engine checks:
1. **Mounted Kaggle Input Cache:** If an already-preprocessed dataset is attached under `/kaggle/input/`, it writes `rsna_pneumonia.yaml` immediately in 1 second!
2. **Working Directory Cache:** If images already exist in `/kaggle/working/rsna_rtdetr_dataset/`, it skips conversion.
3. **Split CSV Cache:** Preserves identical patient groupings.
4. **Auto-Zip Archive:** Automatically generates `/kaggle/working/rsna_rtdetr_dataset.zip` for one-click Kaggle Dataset publishing.


In [5]:
# Cell 5: Main Preprocessing Orchestrator
def run_preprocessing_pipeline(
    label_path=LABEL_PATH,
    class_info_path=CLASS_INFO_PATH,
    input_image_dir=TRAIN_IMAGE_DIR,
    output_dir=OUTPUT_DIR,
    target_size=640,
    max_workers=4
):
    print("=" * 70)
    print("STARTING RSNA PATIENT-LEVEL PREPROCESSING FOR RT-DETR")
    print(f"Target Resolution: {target_size}x{target_size} | Multiprocessing Workers: {max_workers}")
    print("=" * 70)

    split_csv_path = os.path.join(output_dir, "rsna_patient_splits.csv")
    yaml_config_path = os.path.join(output_dir, "rsna_pneumonia.yaml")
    train_img_dir = os.path.join(output_dir, "images", "train")

    # 1. Check if preprocessed dataset is already attached as a Kaggle Input
    attached_candidates = [
        "/kaggle/input/rsna-rtdetr-dataset/rsna_rtdetr_dataset",
        "/kaggle/input/rsna-rtdetr-dataset",
        "/kaggle/input/rsna-rtdetr-preprocessed/rsna_rtdetr_dataset",
        "/kaggle/input/rsna-rtdetr-preprocessed",
        "/kaggle/input/rsna-preprocessed-dataset"
    ]
    for attached in attached_candidates:
        if os.path.exists(attached):
            attached_train = os.path.join(attached, "images", "train")
            if os.path.exists(attached_train) and len(os.listdir(attached_train)) > 0:
                print(f"\n[CACHE HIT - MOUNTED DATASET] Detected preprocessed dataset at: {attached}")
                os.makedirs(output_dir, exist_ok=True)
                with open(yaml_config_path, "w") as f:
                    f.write(f"path: {attached}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n  0: pneumonia\n")
                print(f"YAML config generated referencing mounted input: {yaml_config_path}")
                print("Skipping all DICOM conversions. Ready for immediate training in 1 second!")
                return yaml_config_path

    # 2. Check if already generated in working dir
    if os.path.exists(yaml_config_path) and os.path.exists(train_img_dir) and len(os.listdir(train_img_dir)) > 0:
        print(f"\n[CACHE HIT - WORKING DIR] Dataset already exists at: {output_dir}")
        print("Skipping DICOM conversions. Ready for immediate training!")
        return yaml_config_path

    # 3. Create or load patient-level splits
    if os.path.exists(split_csv_path):
        print(f"\n[CACHE HIT] Found existing split CSV at: {split_csv_path}")
        df_split = pd.read_csv(split_csv_path)
    else:
        df_merged = load_and_merge_metadata(label_path, class_info_path)
        df_split = create_patient_level_splits(df_merged)
        for split in ["train", "val", "test"]:
            os.makedirs(os.path.join(output_dir, "images", split), exist_ok=True)
            os.makedirs(os.path.join(output_dir, "labels", split), exist_ok=True)
        df_split.to_csv(split_csv_path, index=False)
        print(f"Saved split CSV to: {split_csv_path}")

    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(output_dir, "images", split), exist_ok=True)
        os.makedirs(os.path.join(output_dir, "labels", split), exist_ok=True)

    # Save summary metadata JSON
    summary_data = {
        "total_unique_patients": int(df_split["patientId"].nunique()),
        "total_annotation_rows": len(df_split),
        "pneumonia_positive_rows": int((df_split["Target"] == 1).sum()),
        "pneumonia_negative_rows": int((df_split["Target"] == 0).sum()),
        "train_patients": int(df_split[df_split["split"] == "train"]["patientId"].nunique()),
        "val_patients": int(df_split[df_split["split"] == "val"]["patientId"].nunique()),
        "test_patients": int(df_split[df_split["split"] == "test"]["patientId"].nunique()),
        "target_size": target_size,
        "zero_leakage_verified": True
    }
    with open(os.path.join(output_dir, "split_summary.json"), "w") as f:
        json.dump(summary_data, f, indent=2)

    # 4. Multi-processing conversion
    patient_groups = [group for _, group in df_split.groupby("patientId")]
    worker_args = [
        (group.iloc[0]["patientId"], group, input_image_dir, output_dir, target_size)
        for group in patient_groups
    ]

    print(f"\nProcessing {len(worker_args)} patient examinations using {max_workers} CPU workers...")
    processed_count = 0
    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        for result in executor.map(process_single_patient, worker_args):
            if result:
                processed_count += 1
            if processed_count % 2500 == 0 and processed_count > 0:
                print(f"  Progress: {processed_count}/{len(worker_args)} images converted...")

    print(f"Successfully processed and converted {processed_count} patient examinations.")

    # 5. Build RT-DETR YAML
    yaml_content = f"""# RSNA Pneumonia Detection Dataset Config for RT-DETR
path: {output_dir}
train: images/train
val: images/val
test: images/test

names:
  0: pneumonia
"""
    with open(yaml_config_path, "w") as f:
        f.write(yaml_content)
    print(f"Generated RT-DETR YAML at: {yaml_config_path}")

    # 6. Auto-Zip Archive for one-click Kaggle Dataset saving
    zip_dest = os.path.join(os.path.dirname(output_dir), "rsna_rtdetr_dataset")
    zip_file = f"{zip_dest}.zip"
    if not os.path.exists(zip_file):
        print("\n[AUTO-ZIP] Creating compressed archive for one-click Kaggle Dataset creation...")
        shutil.make_archive(base_name=zip_dest, format="zip", root_dir=output_dir)
        print(f"[AUTO-ZIP] Archive successfully created at: {zip_file}")
    else:
        print(f"\n[AUTO-ZIP] Found existing dataset zip at: {zip_file}")

    print("\n" + "=" * 70)
    print("PREPROCESSING COMPLETE! READY FOR RT-DETR TRAINING.")
    print(f"YAML Path: {yaml_config_path}")
    print(f"ZIP Path : {zip_file}")
    print("=" * 70)
    return yaml_config_path


## 5. Run the Preprocessing Pipeline
Set `target_size=640` (recommended for standard Kaggle T4 training) or `target_size=1024` (matches the paper baseline).


In [6]:
# Cell 6: Execute Data Preparation
yaml_config = run_preprocessing_pipeline(
    target_size=640,    # Change to 1024 if matching exact paper resolution
    max_workers=4
)


STARTING RSNA PATIENT-LEVEL PREPROCESSING FOR RT-DETR
Target Resolution: 640x640 | Multiprocessing Workers: 4
Total annotation rows: 30227
Total unique patients: 26684
Pneumonia positive   : 9555 rows
Pneumonia negative   : 20672 rows

--- Patient-Level Split Summary (Zero Leakage Verified) ---
Train patients      : 18678 (70.0%)
Validation patients :  4003 (15.0%)
Test patients       :  4003 (15.0%)
Zero Leakage Check  : Train ∩ Val = 0, Train ∩ Test = 0, Val ∩ Test = 0 [PASSED]
Saved split CSV to: /kaggle/working/rsna_rtdetr_dataset/rsna_patient_splits.csv

Processing 26684 patient examinations using 4 CPU workers...
  Progress: 2500/26684 images converted...
  Progress: 5000/26684 images converted...
  Progress: 7500/26684 images converted...
  Progress: 10000/26684 images converted...
  Progress: 12500/26684 images converted...
  Progress: 15000/26684 images converted...
  Progress: 17500/26684 images converted...
  Progress: 20000/26684 images converted...
  Progress: 22500/26684 

In [7]:
# Cell 7: Verify Preprocessing & Split Summary
summary_file = os.path.join(OUTPUT_DIR, "split_summary.json")
if os.path.exists(summary_file):
    with open(summary_file) as f:
        summary = json.load(f)
    print("=== DATASET PREPROCESSING SUMMARY ===")
    print(json.dumps(summary, indent=2))

train_dir = os.path.join(OUTPUT_DIR, "images", "train")
val_dir = os.path.join(OUTPUT_DIR, "images", "val")
test_dir = os.path.join(OUTPUT_DIR, "images", "test")

train_count = len(os.listdir(train_dir)) if os.path.exists(train_dir) else 0
val_count = len(os.listdir(val_dir)) if os.path.exists(val_dir) else 0
test_count = len(os.listdir(test_dir)) if os.path.exists(test_dir) else 0

print(f"\nGenerated Image Counts:")
print(f"  Train : {train_count} images")
print(f"  Val   : {val_count} images")
print(f"  Test  : {test_count} images")
print(f"  Total : {train_count + val_count + test_count} images")


=== DATASET PREPROCESSING SUMMARY ===
{
  "total_unique_patients": 26684,
  "total_annotation_rows": 30227,
  "pneumonia_positive_rows": 9555,
  "pneumonia_negative_rows": 20672,
  "train_patients": 18678,
  "val_patients": 4003,
  "test_patients": 4003,
  "target_size": 640,
  "zero_leakage_verified": true
}

Generated Image Counts:
  Train : 18678 images
  Val   : 4003 images
  Test  : 4003 images
  Total : 26684 images
